# フーリエ級数近似による周期的波形の不等間隔データ補間

---

## 1. 概要

### 1.1 この資料で扱う問題

製造設備やカム機構の計測では、1サイクルの波形を角度に対して観測したい場面が多くあります。
ただし、実際の計測では次のような事情があり、角度ごとのデータが**等間隔**にそろわないことがあります。

- センサーの取得点数が少ない
- サンプリング角度が毎回少しずれる
- ある角度帯では点が密で、別の角度帯では点が粗い
- 欠測により一部の角度に点がない

このようなとき、

- 不等間隔で得られた観測点から
- 1周期全体のなめらかな波形を作りたい
- その波形をもとに特徴量を計算したい
- 異常検知や比較に使いたい

という要求が出てきます。

本資料では、そのための方法として**フーリエ級数近似**を説明します。

---

### 1.2 フーリエ級数近似で何をしているか

周期的な波形は、

- 1周で1回うねる波
- 1周で2回うねる波
- 1周で3回うねる波

といった、複数の単純な波の足し合わせで表せます。

その単純な波として使うのが、

$$
\cos(\cdot), \quad \sin(\cdot)
$$

です。

したがって、複雑な周期波形を

$$
\text{平均の高さ}
+
\text{基本のうねり}
+
\text{細かいうねり}
+
\cdots
$$

という形で表すことができます。

これがフーリエ級数近似の基本の考え方です。

---

### 1.3 今回の横軸の扱い

今回の横軸は**カム角度**で、範囲は

$$
0^\circ \le x < 360^\circ
$$

とします。

1周期で元に戻る波形を考えるので、

$$
0^\circ \text{ と } 360^\circ
$$

は同じ位置に対応します。

したがって、角度に対する周期関数として扱うのが自然です。

---

### 1.4 この方法の利点

フーリエ級数近似を使う利点は次の通りです。

- 周期性を最初から式に組み込める
- 不等間隔データでもそのまま扱える
- 補間波形を任意の角度で計算できる
- 係数の意味が比較的わかりやすい
- 最小二乗法で計算できるので実装しやすい

特に重要なのは、**不等間隔でも問題なく扱える**点です。
フーリエ変換という言葉から「等間隔データが必要」と思われがちですが、ここで行うのは FFT のような離散変換ではなく、**サイン・コサインを説明変数にした回帰**です。
そのため、各観測角度で三角関数を評価して行列を作れば、そのまま最小二乗法で解けます。

---

## 2. 直感的な数式の理解

### 2.1 基本の近似式

周期波形の近似式を、次のように置きます。

$$
\hat{y}(x)
=
a_0
+
\sum_{k=1}^{K}
\left[
a_k \cos\!\left(\frac{2\pi k x}{360}\right)
+
b_k \sin\!\left(\frac{2\pi k x}{360}\right)
\right]
$$

ここで、

- $x$ はカム角度
- $\hat{y}(x)$ は角度 $x$ における近似波形の値
- $a_0$ は平均の高さ
- $a_k, b_k$ は各成分の係数
- $K$ は何次のうねりまで使うかを決める数

です。

---

### 2.2 各項の意味

まず、

$$
a_0
$$

は波形全体の高さの平均を表します。

次に、

$$
a_1 \cos\!\left(\frac{2\pi x}{360}\right)
+
b_1 \sin\!\left(\frac{2\pi x}{360}\right)
$$

は、1周期の中で1回大きくうねる成分です。

さらに、

$$
a_2 \cos\!\left(\frac{4\pi x}{360}\right)
+
b_2 \sin\!\left(\frac{4\pi x}{360}\right)
$$

は、1周期の中で2回うねる成分です。

同様に、

- $k=3$ なら 1周期で3回うねる成分
- $k=4$ なら 1周期で4回うねる成分

です。

つまり、複雑な波形を「粗いうねり」から「細かいうねり」まで重ね合わせて表現しています。

---

### 2.3 なぜ cos と sin を両方使うのか

1つの周波数成分について、

$$
a_k \cos(k\theta) + b_k \sin(k\theta)
$$

という形は、

$$
R_k \cos(k\theta - \phi_k)
$$

と同じ意味を持ちます。

ここで、

$$
R_k = \sqrt{a_k^2 + b_k^2}
$$

$$
\phi_k = \tan^{-1}\!\left(\frac{b_k}{a_k}\right)
$$

です。

つまり、$\cos$ と $\sin$ を1組で使うことで、その成分について

- どれくらい大きいか
- 山や谷の位置がどれだけずれているか

を表せます。

したがって、$a_k$ と $b_k$ は、振幅と位相の情報を持っていると考えられます。

---

### 2.4 「補間」と「近似」の違い

この方法で求めるのは、まず厳密には**近似式**です。

観測点がノイズを含むとき、求めた式は観測点をすべてぴったり通るとは限りません。
代わりに、全体として誤差が最も小さくなるように係数を決めます。

この意味で、やっていることは

$$
\text{観測点を完全に通す}
$$

のではなく、

$$
\text{観測点に最もよく合う周期関数を求める}
$$

ことです。

ただし、得られた式を使えば任意の角度で値を計算できるため、実務ではその結果を補間波形として利用します。

---

## 3. 行列で書く最小二乗法

### 3.1 観測データの表し方

観測点が $N$ 個あるとします。

$$
(x_1, y_1),\ (x_2, y_2),\ \dots,\ (x_N, y_N)
$$

ここで、

- $x_i$ は角度
- $y_i$ はその角度で観測された値

です。

不等間隔データとは、$x_i$ の間隔が一定でないデータです。

---

### 3.2 未知係数ベクトル

未知の係数をまとめて、次のベクトルで表します。

$$
\boldsymbol{\beta}
=
\begin{bmatrix}
a_0 & a_1 & b_1 & a_2 & b_2 & \cdots & a_K & b_K
\end{bmatrix}^{\mathsf T}
$$

このベクトルの長さは

$$
2K + 1
$$

です。

---

### 3.3 観測ベクトル

観測値を縦に並べると、

$$
\boldsymbol{y}
=
\begin{bmatrix}
y_1 \\
y_2 \\
\vdots \\
y_N
\end{bmatrix}
$$

となります。

---

### 3.4 設計行列

各観測点 $x_i$ に対して、定数項と三角関数の値を並べた行を作ります。

$$
X
=
\begin{bmatrix}
1 & \cos\!\left(\frac{2\pi x_1}{360}\right) & \sin\!\left(\frac{2\pi x_1}{360}\right) & \cdots & \cos\!\left(\frac{2\pi K x_1}{360}\right) & \sin\!\left(\frac{2\pi K x_1}{360}\right)
\\[4pt]
1 & \cos\!\left(\frac{2\pi x_2}{360}\right) & \sin\!\left(\frac{2\pi x_2}{360}\right) & \cdots & \cos\!\left(\frac{2\pi K x_2}{360}\right) & \sin\!\left(\frac{2\pi K x_2}{360}\right)
\\
\vdots & \vdots & \vdots & \ddots & \vdots & \vdots
\\
1 & \cos\!\left(\frac{2\pi x_N}{360}\right) & \sin\!\left(\frac{2\pi x_N}{360}\right) & \cdots & \cos\!\left(\frac{2\pi K x_N}{360}\right) & \sin\!\left(\frac{2\pi K x_N}{360}\right)
\end{bmatrix}
$$

この行列は、

- 行方向が観測点
- 列方向が基底関数

を表しています。

ここで重要なのは、$x_i$ が不等間隔でも何も困らないことです。
単にその角度で $\cos$ と $\sin$ を評価すればよいだけです。

---

### 3.5 モデルの行列表現

すると、近似式はまとめて

$$
\boldsymbol{y} \approx X\boldsymbol{\beta}
$$

と書けます。

これは、通常の線形回帰とまったく同じ形です。
違うのは、説明変数として三角関数を使っていることだけです。

---

### 3.6 最小二乗法の目的関数

求めたいのは、観測値と近似値の差

$$
\boldsymbol{y} - X\boldsymbol{\beta}
$$

の二乗和が最小になる係数です。

したがって、最適化問題は

$$
\hat{\boldsymbol{\beta}}
=
\arg\min_{\boldsymbol{\beta}}
\|\boldsymbol{y} - X\boldsymbol{\beta}\|^2
$$

です。

ここで

$$
\|\boldsymbol{y} - X\boldsymbol{\beta}\|^2
=
\sum_{i=1}^{N}
\left(y_i - \hat{y}(x_i)\right)^2
$$

なので、各観測点での誤差の二乗和を最小にしていることになります。

---

### 3.7 正規方程式

この最小二乗問題の解は、正規方程式

$$
X^{\mathsf T}X\,\hat{\boldsymbol{\beta}}
=
X^{\mathsf T}\boldsymbol{y}
$$

を解くことで得られます。

もし $X^{\mathsf T}X$ が逆行列を持つなら、

$$
\hat{\boldsymbol{\beta}}
=
\left(X^{\mathsf T}X\right)^{-1}X^{\mathsf T}\boldsymbol{y}
$$

です。

これが、フーリエ級数近似を行列形式で最小二乗法により求める基本式です。

---

## 4. 具体的な数値例

ここでは、手計算で追いやすいように **$K=1$** の場合を扱います。
つまり、1周期で1回うねる成分まで使います。

### 4.1 近似式

$$
\hat{y}(x)
=
a_0
+
a_1 \cos\!\left(\frac{2\pi x}{360}\right)
+
b_1 \sin\!\left(\frac{2\pi x}{360}\right)
$$

未知数は

$$
a_0,\ a_1,\ b_1
$$

の3つです。

---

### 4.2 観測データ

不等間隔のカム角度で、次の観測値が得られたとします。

$$
\boldsymbol{x}
=
\begin{bmatrix}
0 \\
45 \\
90 \\
210 \\
300
\end{bmatrix},
\qquad
\boldsymbol{y}
=
\begin{bmatrix}
13.2 \\
12.6 \\
11.1 \\
7.1 \\
10.4
\end{bmatrix}
$$

角度間隔は

$$
45^\circ,\ 45^\circ,\ 120^\circ,\ 90^\circ,\ 60^\circ
$$

であり、等間隔ではありません。

---

### 4.3 設計行列を作る

各角度で $\cos$ と $\sin$ を計算すると、

$$
X
=
\begin{bmatrix}
1 & \cos 0^\circ   & \sin 0^\circ
\\
1 & \cos 45^\circ  & \sin 45^\circ
\\
1 & \cos 90^\circ  & \sin 90^\circ
\\
1 & \cos 210^\circ & \sin 210^\circ
\\
1 & \cos 300^\circ & \sin 300^\circ
\end{bmatrix}
$$

数値で書くと、

$$
X
=
\begin{bmatrix}
1 & 1.0000  & 0.0000
\\
1 & 0.7071  & 0.7071
\\
1 & 0.0000  & 1.0000
\\
1 & -0.8660 & -0.5000
\\
1 & 0.5000  & -0.8660
\end{bmatrix}
$$

です。

未知係数ベクトルは

$$
\boldsymbol{\beta}
=
\begin{bmatrix}
a_0 \\
a_1 \\
b_1
\end{bmatrix}
$$

です。

---

### 4.4 行列の形でモデルを書く

このときモデルは

$$
\begin{bmatrix}
13.2 \\
12.6 \\
11.1 \\
7.1 \\
10.4
\end{bmatrix}
\approx
\begin{bmatrix}
1 & 1.0000  & 0.0000
\\
1 & 0.7071  & 0.7071
\\
1 & 0.0000  & 1.0000
\\
1 & -0.8660 & -0.5000
\\
1 & 0.5000  & -0.8660
\end{bmatrix}
\begin{bmatrix}
a_0 \\
a_1 \\
b_1
\end{bmatrix}
$$

となります。

---

### 4.5 正規方程式を作る

まず、

$$
X^{\mathsf T}X
$$

を計算します。

結果は

$$
X^{\mathsf T}X
=
\begin{bmatrix}
5.0000 & 1.3411 & 0.3411
\\
1.3411 & 2.5000 & 0.5000
\\
0.3411 & 0.5000 & 2.5000
\end{bmatrix}
$$

です。

次に、

$$
X^{\mathsf T}\boldsymbol{y}
$$

を計算すると、

$$
X^{\mathsf T}\boldsymbol{y}
=
\begin{bmatrix}
54.4000
\\
21.1608
\\
7.4529
\end{bmatrix}
$$

です。

したがって正規方程式は

$$
\begin{bmatrix}
5.0000 & 1.3411 & 0.3411
\\
1.3411 & 2.5000 & 0.5000
\\
0.3411 & 0.5000 & 2.5000
\end{bmatrix}
\begin{bmatrix}
a_0 \\
a_1 \\
b_1
\end{bmatrix}
=
\begin{bmatrix}
54.4000
\\
21.1608
\\
7.4529
\end{bmatrix}
$$

です。

---

### 4.6 解を求める

この連立方程式を解くと、

$$
\hat{\boldsymbol{\beta}}
=
\begin{bmatrix}
10.0390 \\
2.8716 \\
1.0372
\end{bmatrix}
$$

となります。

したがって、補間波形は

$$
\hat{y}(x)
=
10.0390
+
2.8716\cos\!\left(\frac{2\pi x}{360}\right)
+
1.0372\sin\!\left(\frac{2\pi x}{360}\right)
$$

です。

これが、任意の角度 $x$ に対して値を返す近似式です。

---

### 4.7 任意角度で補間値を計算する

たとえば、観測点にない

$$
x = 150^\circ
$$

での値を求めてみます。

$$
\hat{y}(150)
=
10.0390
+
2.8716\cos 150^\circ
+
1.0372\sin 150^\circ
$$

$$
=
10.0390
+
2.8716(-0.8660)
+
1.0372(0.5000)
$$

$$
=
10.0390 - 2.4864 + 0.5186
=
8.0712
$$

したがって、

$$
\hat{y}(150^\circ) = 8.0712
$$

です。

このように、元の観測点が存在しない角度でも値を計算できます。

---

### 4.8 振幅と位相で見直す

$$
a_1 \cos\theta + b_1 \sin\theta
$$

は

$$
R\cos(\theta - \phi)
$$

と書けます。

ここで、

$$
R = \sqrt{a_1^2 + b_1^2}
$$

$$
\phi = \tan^{-1}\!\left(\frac{b_1}{a_1}\right)
$$

です。

今回の係数では、

$$
R = \sqrt{2.8716^2 + 1.0372^2} = 3.0532
$$

$$
\phi = \tan^{-1}\!\left(\frac{1.0372}{2.8716}\right) = 0.3466\ \text{rad}
\approx 19.86^\circ
$$

となります。

したがって、近似式は

$$
\hat{y}(x)
=
10.0390
+
3.0532
\cos\!\left(
\frac{2\pi x}{360} - 0.3466
\right)
$$

とも書けます。

この形で見ると、

- 平均の高さは $10.0390$
- 主成分の大きさは $3.0532$
- 山の位置のずれは約 $19.86^\circ$

と読み取れます。

---

## 5. 実務での手順

ここまでの内容を、実務手順としてまとめると次の通りです。

### 5.1 手順 1: 観測データを用意する

1サイクル分の観測データ

$$
(x_i, y_i),\quad i=1,2,\dots,N
$$

を用意します。

ここで $x_i$ はカム角度、$y_i$ はその角度での測定値です。

---

### 5.2 手順 2: 使う高調波数 $K$ を決める

近似式を

$$
\hat{y}(x)
=
a_0
+
\sum_{k=1}^{K}
\left[
a_k \cos\!\left(\frac{2\pi k x}{360}\right)
+
b_k \sin\!\left(\frac{2\pi k x}{360}\right)
\right]
$$

と定めます。

未知数の数は

$$
2K + 1
$$

です。

観測点数 $N$ がこれより十分多くないと、安定した推定が難しくなります。

---

### 5.3 手順 3: 設計行列を作る

各観測角度 $x_i$ について、

$$
1,
\cos\!\left(\frac{2\pi x_i}{360}\right),
\sin\!\left(\frac{2\pi x_i}{360}\right),
\dots,
\cos\!\left(\frac{2\pi K x_i}{360}\right),
\sin\!\left(\frac{2\pi K x_i}{360}\right)
$$

を計算して、行列 $X$ を作ります。

---

### 5.4 手順 4: 最小二乗法で係数を求める

$$
\hat{\boldsymbol{\beta}}
=
\arg\min_{\boldsymbol{\beta}}
\|\boldsymbol{y} - X\boldsymbol{\beta}\|^2
$$

を解きます。

理論式としては

$$
\hat{\boldsymbol{\beta}}
=
\left(X^{\mathsf T}X\right)^{-1}X^{\mathsf T}\boldsymbol{y}
$$

ですが、実装では数値安定性のため、通常は線形方程式ソルバや QR 分解、SVD、あるいは `lstsq` を使います。

---

### 5.5 手順 5: 密な角度列で波形を再構成する

たとえば

$$
0^\circ, 1^\circ, 2^\circ, \dots, 359^\circ
$$

といった密な角度列を用意し、求めた係数を使って

$$
\hat{y}(x)
=
a_0
+
\sum_{k=1}^{K}
\left[
a_k \cos\!\left(\frac{2\pi k x}{360}\right)
+
b_k \sin\!\left(\frac{2\pi k x}{360}\right)
\right]
$$

を計算します。

これにより、なめらかな補間波形が得られます。

---

## 6. なぜ不等間隔でも大丈夫なのか

ここは誤解が多いので、はっきり整理します。

FFT のような離散フーリエ変換は、通常は等間隔サンプリングを前提にしています。
しかし今回やっているのは FFT ではありません。

今回は、

$$
\cos\!\left(\frac{2\pi k x}{360}\right),
\quad
\sin\!\left(\frac{2\pi k x}{360}\right)
$$

を説明変数とした**線形回帰**です。

したがって必要なのは、各観測角度 $x_i$ において、それらの値を計算することだけです。
角度間隔が不規則でも問題ありません。

言い換えると、

- FFT は「等間隔サンプルから係数を高速に求める方法」
- 今回は「任意の観測位置に対して最小二乗で係数を求める方法」

です。

この2つは似て見えますが、前提が違います。

---

## 7. オイラー形式との関係

社内資料や文献によっては、三角関数ではなく複素指数関数で書かれることがあります。

たとえば、オイラーの公式

$$
e^{i\theta} = \cos\theta + i\sin\theta
$$

を用いると、

$$
a_k \cos(k\theta) + b_k \sin(k\theta)
$$

は、複素指数関数を使ってまとめて書けます。

ただし、これは**別の理論**ではありません。
三角関数表示とオイラー表示は同じ内容を表しています。

今回のように、

- 直感的に理解したい
- 行列で最小二乗法を組みたい
- 実数でそのまま実装したい

という目的では、三角関数で書いた方がわかりやすいです。

---

## 8. 実務での注意点

### 8.1 $K$ を大きくしすぎない

高調波数 $K$ を大きくしすぎると、観測点の細かい揺れやノイズまで拾いやすくなります。
その結果、見かけ上は観測点によく合っても、補間波形として不自然になることがあります。

したがって、$K$ は

- 観測点数
- 波形の複雑さ
- ノイズ量

を見ながら決める必要があります。

---

### 8.2 観測点数との関係

未知係数の数は $2K+1$ です。
したがって、少なくとも

$$
N > 2K+1
$$

でないと、十分な自由度がありません。

実務では、単にギリギリよりも、余裕を持って観測点数がある方が安定します。

---

### 8.3 局所スパイクへの弱さ

フーリエ級数近似は、全体としてなめらかな周期波形を表すのが得意です。
一方で、非常に鋭いスパイクや局所的な異常は、低次のフーリエ成分だけでは表しにくいことがあります。

そのため、

- 全体波形の補間はフーリエ級数近似
- スパイク監視は残差や局所特徴量で別途監視

という分担が実務では有効です。

---

### 8.4 周期端の扱いが自然

この方法の大きな利点は、周期端、つまり

$$
0^\circ \text{ 付近と } 360^\circ \text{ 付近}
$$

のつながりが自然なことです。

通常の多項式近似では端点で不自然な形になりやすいですが、周期関数として近似するため、周期端の整合性が取りやすくなります。

---

## 9. まとめ

フーリエ級数近似による不等間隔周期データの補間は、要するに

$$
\text{周期波形をサイン・コサインの和で表し、}
\text{その係数を最小二乗法で求める}
$$

方法です。

今回の重要点は次の通りです。

- 横軸がカム角度 $0^\circ$ から $360^\circ$ の周期データである
- 近似式はサイン・コサインの線形結合で書ける
- 不等間隔データでも設計行列を作ればそのまま扱える
- 係数は最小二乗法で求める
- 得られた式を用いれば任意角度で補間値を計算できる

式として本質を1行で書くと、

$$
\boldsymbol{y} \approx X\boldsymbol{\beta}
$$

であり、やっていることは**線形代数による回帰問題**です。

難しそうに見えても、中身は

- 基底関数を決める
- 行列を作る
- 最小二乗法で係数を求める
- 波形を再構成する

という流れです。

---

## 参考文献

Boyd, S. (2008). *EE263 Course Reader: Least-squares applications*. Stanford University.

Kass, R. E. (n.d.). *Fourier decompositions*. Carnegie Mellon University.

NIST/SEMATECH. (n.d.). *StRD linear least squares regression glossary*. National Institute of Standards and Technology.

Penn State Eberly College of Science. (n.d.). *Topic 2: Time series & autocorrelation*. STAT 501.
